# 01 — Setup y extracción del informe FEMSA 2024

Prepara todo lo demás: monta Drive, clona el repo, instala dependencias con las versiones fijadas del proyecto, descarga el informe anual FEMSA 2024 (edición en inglés, 151 páginas) y extrae el texto por página + las tripletas por reglas. **No necesita Ollama** (la extracción es determinista, sin LLM).

**Orden:** corre este notebook después del 00 (o sin él: este notebook no usa Ollama, pero verifica que el servidor esté arriba para los que siguen).

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Dependencias (versiones fijadas del proyecto)

Se instalan las mismas versiones que usaron las corridas originales (ver `repo/pyproject.toml` y los READMEs de `memo/`, `rag_baseline/` y `agentic_rag/`). En Colab, `torch` se instala sin el índice CPU: pip resuelve la rueda CUDA compatible con la T4. Si Colab ya trae un torch compatible, pip lo deja como está.

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 accelerate>=1.16.0 peft>=0.22.0 sentence-transformers==6.3.0 numpy>=2.5.5 pypdf>=7.1.1 neo4j>=6.4.1 rank-bm25==0.2.2 langchain==1.5.2 langgraph==1.2.24 langchain-core==1.8.5 langchain-huggingface==1.3.0 langchain-ollama

## Verificar Ollama (opcional aquí, obligatorio después)

Este notebook no usa Ollama, pero los notebooks 04–08 sí. Si el servidor no está arriba, corre el notebook `00_ollama_setup.ipynb` (o deja que cada notebook levante el suyo con su bloque idempotente).

In [ ]:
import urllib.request

try:
    with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
        print("Ollama responde en localhost:11434 (status", r.status, ")")
except Exception as e:
    print("Ollama NO responde:", e)
    print("Corre 00_ollama_setup.ipynb, o sigue: los notebooks 04-08 levantan el servidor solos.")

## Descargar el PDF y extraer el texto

Corre `pipeline/extract.py` tal cual (mismo script del repo, sin cambios). Intenta primero la edición en español y luego la de inglés: el sitio en español está detrás de un WAF (Imperva/Incapsula) que bloquea descargas directas, así que en la práctica usa la edición en inglés — mismo informe, mismas cifras. Escribe `pipeline/data/chunks.jsonl` (151 páginas) y las 13 tripletas por reglas.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "extract.py"], cwd=REPO + "/pipeline")
print("exit:", r.returncode)

## Verificación: 151 páginas y la cifra clave

In [ ]:
import json

chunks = [json.loads(l) for l in open(REPO + "/pipeline/data/chunks.jsonl")]
print("paginas:", len(chunks), "| con texto:", sum(1 for c in chunks if c["chars"] > 0))
hits = [c["page"] for c in chunks if "781,585" in c["text"]]
print("781,585 aparece en paginas:", hits)
print("ejemplo p.11:", chunks[10]["text"][:300].replace("\n", " "))